In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
! pip install optuna
! pip install catboost
! pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 8.2 MB/s eta 0:00:00


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import lightgbm as lgb
import optuna
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold, TimeSeriesSplit


TRAIN_PATH = "/content/drive/MyDrive/MUFG/train.csv"
TEST_PATH = "/content/drive/MyDrive/MUFG/test.csv"

SUB_PATH = "/content/drive/MyDrive/MUFG/submit_probs.csv"

LABEL_SUB_PATH = "/content/drive/MyDrive/MUFG/submission_1.csv"
LABEL_THRESHOLD = "auto"

FORCE_TARGET = "LoanStatus"   # 例: "LoanStatus" / None
FORCE_ID = None                # 例: "id" / None

# 学習設定
N_SPLITS = 5
SEED = 42
N_TRIALS = 50
EARLY_STOPPING_ROUNDS = 200
NUM_BOOST_ROUND = 5000

THR_METRIC = "f1"  # "f1" or "acc"

GROUP_COL = "customer_id"
TIME_COL = None

TARGET_ENCODE_COLS = ["NaicsSector", "Subprogram", "BusinessType", "BusinessAge"]


In [ ]:
def set_global_seed(seed: int = 42):
    import os, random
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_global_seed(SEED)


In [ ]:
def guess_id_column(train: pd.DataFrame, test: pd.DataFrame, force_id: str = None):

    if force_id is not None:
        if force_id in test.columns:
            return force_id
        raise ValueError(f"FORCE_ID='{force_id}' が test に存在しません。")

    # 名前で優先検出
    lowmap = {c.lower(): c for c in test.columns}
    for key in ["id", "loan_id", "customer_id", "client_id", "userid", "application_id"]:
        if key in lowmap:
            c = lowmap[key]
            if (not test[c].isna().any()) and (test[c].nunique(dropna=True) == len(test)):
                return c

    candidates = []
    for c in test.columns:
        if test[c].isna().any():
            continue
        nunq = test[c].nunique(dropna=True)
        ratio = nunq / max(1, len(test))
        if ratio >= 0.999:  
            candidates.append((ratio, c))
    candidates.sort(reverse=True)
    if candidates:
        return candidates[0][1]

    return None



def guess_target_column(train: pd.DataFrame, force_target: str = None):
    
    if force_target is not None:
        if force_target in train.columns:
            return force_target
        else:
            raise ValueError(f"FORCE_TARGET='{force_target}' が train に存在しません。")

    name_candidates = [
        "LoanStatus", "loan_status", "is_default", "default", "Default", "label", "Label", "target", "Target"
    ]
    for c in name_candidates:
        if c in train.columns:
            return c

    for c in train.columns:
        vals = train[c].dropna().unique()
        if 2 <= len(vals) <= 4:
            if np.issubdtype(train[c].dtype, np.number):
                s = set(pd.Series(vals).astype(float).round().tolist())
                if s.issubset({0.0, 1.0}) or len(s) == 2:
                    return c
            else:
                return c
    raise ValueError("目的変数の推定に失敗しました。")




In [ ]:
def get_cv_splits(X, y, n_splits=5, seed=42, groups=None, time_col=None):
    if groups is not None:
        gkf = GroupKFold(n_splits=n_splits)
        yield from gkf.split(X, y, groups=groups)
        return

    if time_col is not None and time_col in X.columns:
        order = np.argsort(X[time_col].values)            
        tss = TimeSeriesSplit(n_splits=n_splits)
        for tr, va in tss.split(order):
            yield order[tr], order[va]
        return

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    yield from skf.split(X, y)

In [ ]:
def preprocess_datetime(
    df: pd.DataFrame,
    protected_cols=None,
    verbose=True,
    min_parse_rate=0.6
) -> pd.DataFrame:
    if protected_cols is None:
        protected_cols = set()
    else:
        protected_cols = set(protected_cols)

    dt_cols = []
    for c in df.columns:
        if c in protected_cols:
            continue
        low = c.lower()
        name_hint = (
            ("date" in low) or ("time" in low) or ("timestamp" in low) or ("datetime" in low)
            or low.endswith("_at") or low in {"created_at", "updated_at", "issued_at"}
        )
        try_parse = name_hint or (df[c].dtype == "object")
        if not try_parse:
            continue
        sample = df[c].dropna().astype(str).head(50)
        if len(sample) == 0:
            continue
        parsed = pd.to_datetime(sample, errors="coerce", utc=False)
        parse_rate = parsed.notna().mean()
        if (name_hint and parse_rate >= 0.3) or (parse_rate >= min_parse_rate):
            dt_cols.append(c)

    if verbose and dt_cols:
        print(f"[INFO] datetime-like columns detected: {dt_cols}")

    for c in dt_cols:
        tmp = pd.to_datetime(df[c], errors="coerce", utc=False)
        df[f"{c}_year"]  = tmp.dt.year
        df[f"{c}_month"] = tmp.dt.month
        df[f"{c}_day"]   = tmp.dt.day
        df[f"{c}_dow"]   = tmp.dt.weekday
        df[f"{c}_hour"]  = tmp.dt.hour
        df.drop(columns=[c], inplace=True)

    return df


def drop_high_cardinality_object(
    df: pd.DataFrame,
    protected_cols=None,
    max_unique_ratio: float = 0.9
) -> pd.DataFrame:
    if protected_cols is None:
        protected_cols = set()
    else:
        protected_cols = set(protected_cols)

    drop_cols = []
    for c in df.columns:
        if c in protected_cols:
            continue
        if (df[c].dtype == "object") or (str(df[c].dtype) == "category"):
            nunq = df[c].nunique(dropna=True)
            ratio = nunq / max(1, len(df))
            if ratio >= max_unique_ratio:
                drop_cols.append(c)
    if drop_cols:
        df.drop(columns=drop_cols, inplace=True)
    return df


In [ ]:
def _coerce_binary_target(s: pd.Series) -> pd.Series:
    if np.issubdtype(s.dtype, np.number):
        vals = np.unique(s.dropna().values)
        if set(vals).issubset({0, 1, 0.0, 1.0}):
            out = s.astype(float)
            if out.isna().any():
                mode_val = int(out.dropna().round().mode().iloc[0])
                out = out.fillna(mode_val)
            return out.round().astype(int)
        if len(vals) == 2:
            lo, hi = float(np.min(vals)), float(np.max(vals))
            mapped = s.map({lo: 0, hi: 1})
            if mapped.isna().any():
                mode_val = int(mapped.dropna().mode().iloc[0])
                mapped = mapped.fillna(mode_val)
            return mapped.astype(int)
        raise ValueError("目的変数が2値と解釈できません（数値）。")

    s_str = s.astype(str).str.strip().str.lower()
    mapping = {
        "1": 1, "0": 0,
        "y": 1, "n": 0,
        "yes": 1, "no": 0,
        "true": 1, "false": 0,
        "t": 1, "f": 0,
        "default": 1, "non-default": 0,
        "defaulter": 1, "nondefaulter": 0,
        "bad": 1, "good": 0,
    }
    mapped = s_str.map(mapping)
    cover = mapped.notna().mean()
    if cover >= 0.9:
        majority = int(mapped.dropna().mode().iloc[0])
        return mapped.fillna(majority).astype(int)

    uniq = s_str.dropna().unique()
    if len(uniq) == 2:
        cat = pd.Categorical(s_str)
        codes = pd.Series(cat.codes, index=s.index)
        min_code, max_code = codes.min(), codes.max()
        norm = codes.map({min_code: 0, max_code: 1})
        majority = int(norm.dropna().mode().iloc[0])
        return norm.fillna(majority).astype(int)

    raise ValueError("目的変数が2値に解釈できませんでした。FORCE_TARGET を指定してください。")



def best_threshold(y_true, proba, metric="f1"):
    y_true_np = np.asarray(y_true)
    thr_candidates = np.linspace(0.05, 0.95, 181)
    best_t, best_s = 0.5, -1.0
    for t in thr_candidates:
        y_pred = (proba >= t).astype(int)
        if metric == "f1":
            s = f1_score(y_true_np, y_pred, zero_division=0)
        else:
            s = accuracy_score(y_true_np, y_pred)
        if s > best_s:
            best_s, best_t = s, t
    return float(best_t), float(best_s)



def _align_train_test_columns(train_df: pd.DataFrame, test_df: pd.DataFrame, fill_value=0):
    train_aligned = train_df.copy()
    test_aligned = test_df.copy()

    missing_in_test = [c for c in train_aligned.columns if c not in test_aligned.columns]
    for col in missing_in_test:
        test_aligned[col] = fill_value

    missing_in_train = [c for c in test_aligned.columns if c not in train_aligned.columns]
    for col in missing_in_train:
        train_aligned[col] = fill_value

    return train_aligned, test_aligned[train_aligned.columns]



def _cast_columns_as_object(df: pd.DataFrame, columns) -> pd.DataFrame:
    df = df.copy()
    for col in columns:
        if col in df.columns:
            df[col] = df[col].astype(object)
    return df



def _apply_smoothed_target_encoding(
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    *frames,
    columns=None,
    smoothing_m: int = 20,
    target_name: str = "LoanStatus",
):
    te_cols = TARGET_ENCODE_COLS if columns is None else columns
    used_cols = [col for col in te_cols if col in X_tr.columns]

    encoded_frames = [X_tr.copy(), *(frame.copy() for frame in frames)]
    if not used_cols:
        return encoded_frames, used_cols

    global_target_mean = y_tr.mean()
    for col in used_cols:
        temp_df = pd.DataFrame({col: X_tr[col], target_name: y_tr})
        agg = temp_df.groupby(col)[target_name].agg(["mean", "count"])
        smoothed_mean = (
            agg["count"] * agg["mean"] + smoothing_m * global_target_mean
        ) / (agg["count"] + smoothing_m)

        for frame in encoded_frames:
            frame[f"{col}_te"] = frame[col].map(smoothed_mean).fillna(global_target_mean)

    processed_frames = [frame.drop(columns=used_cols) for frame in encoded_frames]
    return processed_frames, used_cols


In [ ]:
def prepare_data(
    train_path: str,
    test_path: str,
    force_target: str = None,
    force_id: str = None,
    verbose: bool = True,
):
    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    target_col = guess_target_column(train, force_target=force_target)
    id_col_test = guess_id_column(train, test, force_id=force_id)
    id_col_train = id_col_test if id_col_test in train.columns else None

    train["_is_train"] = 1
    test["_is_train"] = 0
    original_y = train[target_col].copy()

    if target_col not in test.columns:
        test[target_col] = -1

    all_df = pd.concat([train, test], axis=0, ignore_index=True)

    print("[INFO] Creating final aggregation and interaction features...")

    business_age_map = {
        "Startup, Loan Funds will Open Business": 0,
        "New Business or 2 years or less": 1,
        "Existing or more than 2 years old": 2,
        "Change of Ownership": 2,
        "Unanswered": 1,
    }
    all_df["BusinessAge_ordinal"] = all_df["BusinessAge"].map(business_age_map)
    all_df["CollateralInd"] = all_df["CollateralInd"].map({"Y": 1, "N": 0})
    all_df["FixedOrVariableInterestInd"] = all_df["FixedOrVariableInterestInd"].map({"V": 1, "F": 0})

    all_df["SBA_Portion"] = all_df["SBAGuaranteedApproval"] / (all_df["GrossApproval"] + 1e-6)
    all_df["Approval_per_Job"] = all_df["GrossApproval"] / (all_df["JobsSupported"] + 1)
    all_df["Years_to_Pay"] = all_df["TermInMonths"] / 12

    agg_keys = ["NaicsSector", "BusinessAge", "Subprogram", "BusinessType"]
    agg_targets = ["InitialInterestRate", "GrossApproval", "SBA_Portion", "TermInMonths"]

    for key in agg_keys:
        for target in agg_targets:
            group_stats = all_df.groupby(key)[target].agg(["mean", "std", "max", "min"]).reset_index()
            group_stats.columns = [
                key,
                f"{key}_{target}_mean",
                f"{key}_{target}_std",
                f"{key}_{target}_max",
                f"{key}_{target}_min",
            ]
            all_df = pd.merge(all_df, group_stats, on=key, how="left")
            all_df[f"{target}_vs_{key}_mean_diff"] = all_df[target] - all_df[f"{key}_{target}_mean"]
            all_df[f"{target}_vs_{key}_mean_ratio"] = all_df[target] / (all_df[f"{key}_{target}_mean"] + 1e-6)

    print(f"[SUCCESS] Added {len(agg_keys) * len(agg_targets) * 6} aggregation features!")

    protected_cols = {target_col, "_is_train"}
    for col in [id_col_train, id_col_test]:
        if col:
            protected_cols.add(col)

    all_df = preprocess_datetime(all_df, protected_cols=protected_cols, verbose=verbose)
    all_df = drop_high_cardinality_object(all_df, protected_cols=protected_cols)

    for col in all_df.select_dtypes(include="object").columns:
        if col not in protected_cols:
            all_df[col] = all_df[col].astype("category")

    train = all_df[all_df["_is_train"] == 1].drop(columns=["_is_train"])
    test = all_df[all_df["_is_train"] == 0].drop(columns=["_is_train", target_col])
    train[target_col] = original_y

    feature_cols = [col for col in train.columns if col not in protected_cols]
    cat_cols = [col for col in feature_cols if str(train[col].dtype) in ["category", "bool", "object"]]

    X = train[feature_cols].copy()
    y = _coerce_binary_target(train[target_col].copy()).astype(int)
    X_test = test[feature_cols].copy()

    if set(X.columns) != set(X_test.columns):
        print("Warning: Train and test columns do not match perfectly.")
        X, X_test = _align_train_test_columns(X, X_test)

    test_ids = test[id_col_test].copy() if id_col_test in test.columns else pd.Series(range(len(test)), name="id")
    groups = train[GROUP_COL].values if GROUP_COL not in [None, ""] and GROUP_COL in train.columns else None

    return X, y, X_test, test_ids, feature_cols, cat_cols, target_col, id_col_test, groups


In [ ]:
def optimize_hyperparams(
    X, y, cat_cols,
    n_splits=5,
    seed=42,
    n_trials=50,
    early_stopping_rounds=200,
    num_boost_round=5000,
    groups=None,
    time_col=None,
):
    pos = int((y == 1).sum())
    neg = int((y == 0).sum())
    base_pos_weight = neg / max(1, pos)
    print(f"[INFO] class balance: pos={pos}, neg={neg}, base_pos_weight≈{base_pos_weight:.3f}")
    print(f"[INFO] threshold metric = {THR_METRIC}")

    X_obj = _cast_columns_as_object(X, TARGET_ENCODE_COLS)

    def cv_objective(trial: optuna.trial.Trial):
        params = {
            "objective": "binary",
            "metric": "auc",
            "boosting_type": "gbdt",
            "verbosity": -1,
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08),
            "num_leaves": trial.suggest_int("num_leaves", 16, 256, log=True),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 200),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 0, 10),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
            "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 1.0),
            "max_depth": trial.suggest_int("max_depth", -1, 12),
            "scale_pos_weight": base_pos_weight * trial.suggest_float("pos_weight_mult", 0.5, 4.0),
            "seed": seed,
            "n_jobs": -1,
        }

        cv_iter = get_cv_splits(X, y, n_splits=n_splits, seed=seed, groups=groups, time_col=time_col)
        aucs, metric_scores, best_rounds, thrs = [], [], [], []

        for tr_idx, va_idx in cv_iter:
            X_tr = X_obj.iloc[tr_idx].copy()
            X_va = X_obj.iloc[va_idx].copy()
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            processed_frames, used_te_cols = _apply_smoothed_target_encoding(
                X_tr,
                y_tr,
                X_va,
                columns=TARGET_ENCODE_COLS,
            )
            X_tr_processed, X_va_processed = processed_frames
            remaining_cat_cols = [col for col in cat_cols if col not in used_te_cols]

            dtr = lgb.Dataset(X_tr_processed, label=y_tr, categorical_feature=remaining_cat_cols)
            dva = lgb.Dataset(X_va_processed, label=y_va, categorical_feature=remaining_cat_cols)

            model = lgb.train(
                params,
                dtr,
                num_boost_round=num_boost_round,
                valid_sets=[dtr, dva],
                valid_names=["train", "valid"],
                callbacks=[
                    lgb.early_stopping(early_stopping_rounds, verbose=False),
                    lgb.log_evaluation(0),
                ],
            )

            p_va = model.predict(X_va_processed, num_iteration=model.best_iteration)
            aucs.append(roc_auc_score(y_va, p_va))

            thr_best, score_best = best_threshold(y_va.values, p_va, metric=THR_METRIC)
            metric_scores.append(score_best)
            best_rounds.append(model.best_iteration)
            thrs.append(thr_best)

        mean_auc = float(np.mean(aucs))
        mean_metric = float(np.mean(metric_scores))
        mean_round = int(np.mean(best_rounds))
        mean_thr = float(np.mean(thrs))

        score = mean_metric if THR_METRIC == "f1" else mean_auc
        trial.set_user_attr("cv_auc", mean_auc)
        trial.set_user_attr("cv_metric", mean_metric)
        trial.set_user_attr("best_round", mean_round)
        trial.set_user_attr("best_thr", mean_thr)
        return score

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(cv_objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_trial
    best_params = best.params.copy()
    cv_auc = best.user_attrs["cv_auc"]
    cv_metric = best.user_attrs["cv_metric"]
    best_round = best.user_attrs["best_round"]
    best_thr = best.user_attrs["best_thr"]

    print()
    print("========== Best Trial ==========")
    print(f"AUC (CV mean): {cv_auc:.4f}")
    label_metric = "F1" if THR_METRIC == "f1" else "ACC"
    print(f"{label_metric} (CV mean, tuned thr): {cv_metric:.4f} @thr≈{best_thr:.3f}")
    print(f"best_round≈{best_round}")
    print("best params:")
    for key, value in best_params.items():
        print(f"  {key}: {value}")

    final_params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "verbosity": -1,
        "seed": seed,
        "n_jobs": -1,
        **best_params,
    }
    final_params["scale_pos_weight"] = base_pos_weight * final_params.pop("pos_weight_mult")

    return final_params, best_round, best_thr, cv_auc, cv_metric


In [ ]:
import xgboost as xgb
from sklearn.linear_model import LogisticRegression


def _prepare_xgboost_inputs(X, X_test=None):
    X_xgb = pd.get_dummies(X, dummy_na=True)
    if X_test is None:
        return X_xgb

    X_test_xgb = pd.get_dummies(X_test, dummy_na=True)
    return _align_train_test_columns(X_xgb, X_test_xgb)



def optimize_hyperparams_xgboost(
    X, y,
    n_splits=5,
    seed=42,
    n_trials=30,
    groups=None,
):
    print("[INFO][XGBoost] Starting hyperparameter optimization...")

    X_xgb = _prepare_xgboost_inputs(X)
    scale_pos_weight = (y == 0).sum() / max(1, (y == 1).sum())

    def objective(trial):
        params = {
            "objective": "binary:logistic",
            "eval_metric": "auc",
            "booster": "gbtree",
            "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "eta": trial.suggest_float("eta", 0.01, 0.1, log=True),
            "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "scale_pos_weight": scale_pos_weight,
            "seed": seed,
            "nthread": -1,
        }

        cv_iter = get_cv_splits(X_xgb, y, n_splits=n_splits, seed=seed, groups=groups)
        aucs = []

        for tr_idx, va_idx in cv_iter:
            X_tr, X_va = X_xgb.iloc[tr_idx], X_xgb.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

            dtrain = xgb.DMatrix(X_tr, label=y_tr)
            dvalid = xgb.DMatrix(X_va, label=y_va)

            model = xgb.train(
                params,
                dtrain,
                num_boost_round=2000,
                evals=[(dvalid, "valid")],
                early_stopping_rounds=100,
                verbose_eval=False,
            )
            preds = model.predict(dvalid, iteration_range=(0, model.best_iteration))
            aucs.append(roc_auc_score(y_va, preds))

        return np.mean(aucs)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    print()
    print("========== [XGBoost] Best Trial ==========")
    print(f"AUC (CV mean): {study.best_value:.4f}")
    print("Best params:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    final_params = study.best_params.copy()
    final_params["objective"] = "binary:logistic"
    final_params["eval_metric"] = "auc"
    final_params["seed"] = seed
    final_params["scale_pos_weight"] = scale_pos_weight

    return final_params



def train_cv_and_predict_xgboost(
    X, y, X_test,
    xgb_params,
    n_splits=5,
    seed=42,
    groups=None,
):
    print("[INFO][XGBoost] Starting final CV training and prediction...")

    X_xgb, X_test_xgb = _prepare_xgboost_inputs(X, X_test)
    cv_iter = get_cv_splits(X_xgb, y, n_splits=n_splits, seed=seed, groups=groups)
    oof_pred_proba = np.zeros(len(X))
    test_pred_proba = np.zeros(len(X_test))
    dtest = xgb.DMatrix(X_test_xgb)

    for fold, (tr_idx, va_idx) in enumerate(cv_iter, 1):
        X_tr, y_tr = X_xgb.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X_xgb.iloc[va_idx], y.iloc[va_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dvalid = xgb.DMatrix(X_va, label=y_va)

        model = xgb.train(
            xgb_params,
            dtrain,
            num_boost_round=5000,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=150,
            verbose_eval=False,
        )

        oof_pred_proba[va_idx] = model.predict(dvalid, iteration_range=(0, model.best_iteration))
        test_pred_proba += model.predict(dtest, iteration_range=(0, model.best_iteration)) / n_splits

    return test_pred_proba, oof_pred_proba


In [ ]:
def train_cv_and_predict(
    X, y, X_test, cat_cols,
    lgb_params,
    best_round,
    n_splits=5,
    seed=42,
    early_stopping_rounds=200,
    groups=None,
    time_col=None,
):
    cv_iter = get_cv_splits(X, y, n_splits=n_splits, seed=seed, groups=groups, time_col=time_col)
    oof_pred = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    fold_scores = []

    te_cols = [col for col in TARGET_ENCODE_COLS if col in X.columns]
    processed_X_columns = X.drop(columns=te_cols).columns
    feature_importances = pd.DataFrame(index=processed_X_columns)

    X_obj = _cast_columns_as_object(X, te_cols)
    X_test_obj = _cast_columns_as_object(X_test, te_cols)

    print("[INFO] Starting CV with Target Encoding...")
    for fold, (tr_idx, va_idx) in enumerate(cv_iter, 1):
        X_tr = X_obj.iloc[tr_idx].copy()
        X_va = X_obj.iloc[va_idx].copy()
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        processed_frames, used_te_cols = _apply_smoothed_target_encoding(
            X_tr,
            y_tr,
            X_va,
            X_test_obj,
            columns=te_cols,
        )
        X_tr_processed, X_va_processed, X_test_processed = processed_frames
        remaining_cat_cols = [col for col in cat_cols if col not in used_te_cols]

        dtr = lgb.Dataset(X_tr_processed, label=y_tr, categorical_feature=remaining_cat_cols)
        dva = lgb.Dataset(X_va_processed, label=y_va, categorical_feature=remaining_cat_cols)

        model = lgb.train(
            lgb_params,
            dtr,
            num_boost_round=max(100, int(best_round * 1.2)),
            valid_sets=[dtr, dva],
            valid_names=["train", "valid"],
            callbacks=[
                lgb.early_stopping(early_stopping_rounds, verbose=False),
                lgb.log_evaluation(0),
            ],
        )

        feature_importances[f"fold_{fold}"] = pd.Series(
            model.feature_importance(importance_type="gain"),
            index=X_tr_processed.columns,
        )

        p_va = model.predict(X_va_processed, num_iteration=model.best_iteration)
        oof_pred[va_idx] = p_va
        test_pred += model.predict(X_test_processed, num_iteration=model.best_iteration) / n_splits

        _, score_best = best_threshold(y_va.values, p_va, metric=THR_METRIC)
        fold_scores.append(score_best)

    oof_auc = roc_auc_score(y, oof_pred)
    global_thr, _ = best_threshold(y.values, oof_pred, metric=THR_METRIC)
    oof_metric = f1_score(y, (oof_pred >= global_thr).astype(int))

    label_metric = "F1" if THR_METRIC == "f1" else "ACC"
    print()
    print("========== CV Summary ==========")
    print(f"OOF AUC: {oof_auc:.4f}")
    print(f"OOF {label_metric} (thr={global_thr:.3f}): {oof_metric:.4f}")
    print(f"Fold {label_metric}s: {[round(score, 4) for score in fold_scores]}")

    return test_pred, oof_pred, global_thr, oof_auc, oof_metric, feature_importances


In [ ]:
def save_submission(test_ids, preds, path: str, id_col_name: str = "id"):
    """
    確率提出（解析用）：ヘッダあり ['id', 'score']
    """
    out = pd.DataFrame({id_col_name: test_ids, "score": preds})
    out.to_csv(path, index=False)
    print(f"[INFO] probability submission saved to: {path}")


def save_submission_label_noheader(test_ids, preds, path: str, thr: float):
    """
    本番提出: ヘッダ無し2列（id, label）
    - IDは必ず整数 & 欠損/重複なしを厳格チェック
    - ここでNGなら ValueError を出して気づけるようにする
    """
    ids = pd.Series(test_ids).copy()

    # 文字列 "nan" や float NaN を完全排除
    if ids.isna().any():
        na_idx = list(np.where(ids.isna())[0][:5])
        raise ValueError(f"[SUBMIT-ERROR] test_ids に NaN があります（例: 行index {na_idx}）。FORCE_ID を見直してください。")

    # 数値化（小数->整数）を強制。失敗したら明示エラーで止める
    try:
        ids_num = pd.to_numeric(ids, errors="raise")
    except Exception as e:
        raise ValueError(f"[SUBMIT-ERROR] ID を数値化できません。文字列や混在が含まれています。FORCE_ID を正しく指定してください。詳細: {e}")

    # 小数が混じっていたら整数へ（例: 7553.0 -> 7553）
    if (ids_num % 1 != 0).any():
        # 端数が出るのは ID 列推定ミスの可能性大
        bad_idx = list(np.where(ids_num % 1 != 0)[0][:5])
        raise ValueError(f"[SUBMIT-ERROR] 非整数のIDが含まれています（例: 行index {bad_idx}）。ID列の推定が誤っている可能性。FORCE_ID 再設定を。")

    ids_int = ids_num.astype(np.int64)

    # 重複チェック
    dup_mask = ids_int.duplicated(keep=False)
    if dup_mask.any():
        dup_vals = ids_int[dup_mask].unique()[:5]
        raise ValueError(f"[SUBMIT-ERROR] ID の重複があります（例: {dup_vals.tolist()}）。FORCE_ID を再確認してください。")

    # ラベル二値化
    label = (pd.Series(preds) >= float(thr)).astype(int)

    # 出力（※ヘッダ無し）
    out = pd.DataFrame({"id": ids_int, "label": label})
    out.to_csv(path, index=False, header=False)
    print(f"[INFO] headerless label submission saved to: {path} (thr={thr:.3f})")



def save_preview_labels(test_ids, preds, thr: float, id_col_name: str = "id", path: str = None, head: int = 10):
    """
    確率と予測ラベルのプレビュー（ヘッダあり、先頭N件を表示/保存）
    """
    df = pd.DataFrame({
        id_col_name: test_ids,
        "proba": preds,
        "pred_label": (np.asarray(preds) >= float(thr)).astype(int)
    })
    if path:
        df.to_csv(path, index=False)
        print(f"[INFO] preview saved to: {path}")
    display(df.head(head))


In [ ]:
def plot_feature_importance(df_importance, top_n=30):
    """特徴量重要度をプロットする"""
    # Fold全体の平均と標準偏差を計算
    df_importance['mean'] = df_importance.mean(axis=1)
    df_importance['std'] = df_importance.std(axis=1)

    # 上位top_n件を抽出してプロット
    plt.figure(figsize=(12, max(6, top_n // 2)))
    sns.barplot(
        x=df_importance.sort_values('mean', ascending=False)['mean'].head(top_n),
        y=df_importance.sort_values('mean', ascending=False).head(top_n).index,
        palette='viridis'
    )
    plt.title(f'Top {top_n} Feature Importances (Gain)')
    plt.xlabel('Mean Importance (Gain)')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.savefig("feature_importance.png") # ファイルに保存
    plt.show()

In [ ]:
def run_pipeline():
    X, y, X_test, test_ids, _, cat_cols, _, id_col_for_submit, groups = load_prepared_data(verbose=True)

    lgb_result = train_lightgbm_model(
        X,
        y,
        X_test,
        cat_cols,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=N_TRIALS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        num_boost_round=NUM_BOOST_ROUND,
        groups=groups,
        time_col=TIME_COL,
    )

    plot_feature_importance(lgb_result["importances"], top_n=30)

    save_submission(test_ids, lgb_result["test_pred"], SUB_PATH, id_col_name=id_col_for_submit)

    thr_to_use = (
        lgb_result["thr"]
        if isinstance(LABEL_THRESHOLD, str) and LABEL_THRESHOLD.lower() == "auto"
        else float(LABEL_THRESHOLD)
    )
    save_submission_label_noheader(test_ids, lgb_result["test_pred"], LABEL_SUB_PATH, thr=thr_to_use)
    save_preview_labels(test_ids, lgb_result["test_pred"], thr=thr_to_use, id_col_name=id_col_for_submit)

    label_metric = "F1" if THR_METRIC == "f1" else "ACC"
    print()
    print("========== FINAL ==========")
    print(
        f"CV(Best) AUC: {lgb_result['cv_auc']:.4f}, "
        f"{label_metric}: {lgb_result['cv_metric']:.4f} (thr≈{lgb_result['best_thr']:.3f})"
    )
    print(
        f"OOF      AUC: {lgb_result['oof_auc']:.4f}, "
        f"{label_metric}: {lgb_result['oof_metric']:.4f} (thr≈{lgb_result['thr']:.3f})"
    )
    print(f"[INFO] Submit this (headerless) file: {LABEL_SUB_PATH}")


In [ ]:
from catboost import CatBoostClassifier


def optimize_hyperparams_catboost(
    X, y, cat_cols,
    n_splits=5,
    seed=42,
    n_trials=30,
    groups=None,
):
    print("[INFO][CatBoost] Starting hyperparameter optimization...")

    def objective(trial):
        params = {
            "objective": "Logloss",
            "eval_metric": "F1",
            "iterations": trial.suggest_int("iterations", 500, 2000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
            "depth": trial.suggest_int("depth", 4, 10),
            "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
            "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
            "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
            "border_count": trial.suggest_int("border_count", 32, 255),
            "verbose": 0,
            "random_seed": seed,
            "auto_class_weights": "Balanced",
        }

        cv_iter = get_cv_splits(X, y, n_splits=n_splits, seed=seed, groups=groups)
        scores = []

        for tr_idx, va_idx in cv_iter:
            X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
            X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

            model = CatBoostClassifier(**params)
            model.fit(
                X_tr,
                y_tr,
                eval_set=[(X_va, y_va)],
                cat_features=cat_cols,
                early_stopping_rounds=100,
                verbose=0,
            )
            scores.append(model.get_best_score()["validation"]["F1"])

        return np.mean(scores)

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=seed),
    )
    study.optimize(objective, n_trials=n_trials)

    print()
    print("========== [CatBoost] Best Trial ==========")
    print(f"F1 (CV mean): {study.best_value:.4f}")
    print("Best params:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    final_params = study.best_params.copy()
    final_params["random_seed"] = seed
    final_params["verbose"] = 0
    final_params["auto_class_weights"] = "Balanced"

    return final_params



def train_cv_and_predict_catboost(
    X, y, X_test, cat_cols,
    cb_params,
    n_splits=5,
    seed=42,
    groups=None,
):
    print("[INFO][CatBoost] Starting final CV training and prediction...")
    cv_iter = get_cv_splits(X, y, n_splits=n_splits, seed=seed, groups=groups)
    oof_pred_proba = np.zeros(len(X))
    test_pred_proba = np.zeros(len(X_test))

    for fold, (tr_idx, va_idx) in enumerate(cv_iter, 1):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_va, y_va = X.iloc[va_idx], y.iloc[va_idx]

        model = CatBoostClassifier(**cb_params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            cat_features=cat_cols,
            early_stopping_rounds=150,
            verbose=0,
        )

        oof_pred_proba[va_idx] = model.predict_proba(X_va)[:, 1]
        test_pred_proba += model.predict_proba(X_test)[:, 1] / n_splits

    oof_auc = roc_auc_score(y, oof_pred_proba)
    global_thr, _ = best_threshold(y.values, oof_pred_proba, metric="f1")
    oof_f1 = f1_score(y, (oof_pred_proba >= global_thr).astype(int))

    print()
    print("========== [CatBoost] CV Summary ==========")
    print(f"OOF AUC: {oof_auc:.4f}")
    print(f"OOF F1 (thr={global_thr:.3f}): {oof_f1:.4f}")

    return test_pred_proba, oof_pred_proba, global_thr



def run_catboost_pipeline():
    X, y, X_test, test_ids, _, cat_cols, _, _, groups = load_prepared_data(verbose=True)

    cb_result = train_catboost_model(
        X,
        y,
        X_test,
        cat_cols,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=30,
        groups=groups,
    )

    catboost_sub_path = LABEL_SUB_PATH.replace(".csv", "_catboost.csv")
    save_submission_label_noheader(test_ids, cb_result["test_pred"], catboost_sub_path, thr=cb_result["thr"])
    print()
    print(f"[INFO] CatBoost submission saved to: {catboost_sub_path}")


In [ ]:
def run_ensemble_pipeline():
    print("========== Starting Weighted Ensemble Pipeline ==========")

    X, y, X_test, test_ids, _, cat_cols, _, _, groups = load_prepared_data(verbose=False)
    print()
    print("[INFO] Data preparation complete.")

    print()
    print("========== [1/2] Training LightGBM Model ==========")
    lgb_result = train_lightgbm_model(
        X,
        y,
        X_test,
        cat_cols,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=30,
        groups=groups,
    )
    print()
    print("[INFO] LightGBM prediction complete.")

    print()
    print("========== [2/2] Training CatBoost Model ==========")
    cb_result = train_catboost_model(
        X,
        y,
        X_test,
        cat_cols,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=30,
        groups=groups,
    )
    print()
    print("[INFO] CatBoost prediction complete.")

    print()
    print("========== Searching for Optimal Ensemble Weight ==========")
    best_f1 = 0
    best_weight = 0

    for weight in np.arange(0.5, 1.01, 0.01):
        weighted_oof = lgb_result["oof_pred"] * weight + cb_result["oof_pred"] * (1 - weight)
        temp_thr, _ = best_threshold(y, weighted_oof, metric="f1")
        temp_f1 = f1_score(y, (weighted_oof >= temp_thr).astype(int))

        if temp_f1 > best_f1:
            best_f1 = temp_f1
            best_weight = weight

    print()
    print("OOF F1 Scores:")
    print(f"  - LightGBM only: {lgb_result['oof_metric']:.4f}")
    print(f"  - CatBoost only: {cb_result['oof_f1']:.4f}")
    print(f"  - Weighted Ensemble: {best_f1:.4f} (at LGB weight = {best_weight:.2f})")

    ensemble_oof = lgb_result["oof_pred"] * best_weight + cb_result["oof_pred"] * (1 - best_weight)
    ensemble_test_pred = lgb_result["test_pred"] * best_weight + cb_result["test_pred"] * (1 - best_weight)
    final_thr, _ = best_threshold(y, ensemble_oof, metric="f1")

    ensemble_sub_path = LABEL_SUB_PATH.replace(".csv", "_weighted_ensemble.csv")
    save_submission_label_noheader(test_ids, ensemble_test_pred, ensemble_sub_path, thr=final_thr)
    print()
    print(f"[INFO] Weighted Ensemble submission saved to: {ensemble_sub_path} (thr={final_thr:.3f})")
    print()
    print("🎉🎉🎉 Weighted Ensemble pipeline finished successfully! 🎉🎉🎉")


In [ ]:
def load_prepared_data(verbose=False):
    return prepare_data(
        TRAIN_PATH,
        TEST_PATH,
        force_target=FORCE_TARGET,
        force_id=FORCE_ID,
        verbose=verbose,
    )



def train_lightgbm_model(
    X, y, X_test, cat_cols,
    n_splits=5,
    seed=42,
    n_trials=50,
    early_stopping_rounds=200,
    num_boost_round=5000,
    groups=None,
    time_col=None,
):
    best_params, best_round, best_thr, cv_auc, cv_metric = optimize_hyperparams(
        X,
        y,
        cat_cols,
        n_splits=n_splits,
        seed=seed,
        n_trials=n_trials,
        early_stopping_rounds=early_stopping_rounds,
        num_boost_round=num_boost_round,
        groups=groups,
        time_col=time_col,
    )

    test_pred, oof_pred, global_thr, oof_auc, oof_metric, importances = train_cv_and_predict(
        X,
        y,
        X_test,
        cat_cols,
        lgb_params=best_params,
        best_round=best_round,
        n_splits=n_splits,
        seed=seed,
        early_stopping_rounds=early_stopping_rounds,
        groups=groups,
        time_col=time_col,
    )

    return {
        "params": best_params,
        "best_round": best_round,
        "best_thr": best_thr,
        "cv_auc": cv_auc,
        "cv_metric": cv_metric,
        "test_pred": test_pred,
        "oof_pred": oof_pred,
        "thr": global_thr,
        "oof_auc": oof_auc,
        "oof_metric": oof_metric,
        "importances": importances,
    }



def train_catboost_model(
    X, y, X_test, cat_cols,
    n_splits=5,
    seed=42,
    n_trials=30,
    groups=None,
):
    cb_params = optimize_hyperparams_catboost(
        X,
        y,
        cat_cols,
        n_splits=n_splits,
        seed=seed,
        n_trials=n_trials,
        groups=groups,
    )
    test_pred, oof_pred, thr = train_cv_and_predict_catboost(
        X,
        y,
        X_test,
        cat_cols,
        cb_params=cb_params,
        n_splits=n_splits,
        seed=seed,
        groups=groups,
    )

    return {
        "params": cb_params,
        "test_pred": test_pred,
        "oof_pred": oof_pred,
        "thr": thr,
        "oof_auc": roc_auc_score(y, oof_pred),
        "oof_f1": f1_score(y, (oof_pred >= thr).astype(int)),
    }



def train_xgboost_model(
    X, y, X_test,
    n_splits=5,
    seed=42,
    n_trials=30,
    groups=None,
):
    xgb_params = optimize_hyperparams_xgboost(
        X,
        y,
        n_splits=n_splits,
        seed=seed,
        n_trials=n_trials,
        groups=groups,
    )
    test_pred, oof_pred = train_cv_and_predict_xgboost(
        X,
        y,
        X_test,
        xgb_params=xgb_params,
        n_splits=n_splits,
        seed=seed,
        groups=groups,
    )

    return {
        "params": xgb_params,
        "test_pred": test_pred,
        "oof_pred": oof_pred,
        "oof_auc": roc_auc_score(y, oof_pred),
    }



def build_meta_features(**predictions):
    return pd.DataFrame(predictions)


In [ ]:
def run_stacking_pipeline():
    print("========== 👑 Starting Stacking Pipeline with LGBM Meta-Model 👑 ==========")

    X, y, X_test, test_ids, _, cat_cols, _, _, groups = load_prepared_data(verbose=False)
    print()
    print("[INFO] Data preparation complete.")

    print()
    print("========== [Level 0] Training Base Models ==========")
    print()
    print("--- [1/3] Training LightGBM ---")
    lgb_result = train_lightgbm_model(
        X,
        y,
        X_test,
        cat_cols,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=50,
        groups=groups,
    )

    print()
    print("--- [2/3] Training CatBoost ---")
    cb_result = train_catboost_model(
        X,
        y,
        X_test,
        cat_cols,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=50,
        groups=groups,
    )

    print()
    print("--- [3/3] Training XGBoost ---")
    xgb_result = train_xgboost_model(
        X,
        y,
        X_test,
        n_splits=N_SPLITS,
        seed=SEED,
        n_trials=50,
        groups=groups,
    )
    print()
    print("[INFO] All base model predictions are ready.")

    print()
    print("========== [Level 1] Training LightGBM Meta-Model ==========")
    meta_X = build_meta_features(
        lgb_pred=lgb_result["oof_pred"],
        cb_pred=cb_result["oof_pred"],
        xgb_pred=xgb_result["oof_pred"],
    )
    meta_X_test = build_meta_features(
        lgb_pred=lgb_result["test_pred"],
        cb_pred=cb_result["test_pred"],
        xgb_pred=xgb_result["test_pred"],
    )

    meta_lgb_params = {
        "objective": "binary",
        "metric": "logloss",
        "boosting_type": "gbdt",
        "n_estimators": 200,
        "learning_rate": 0.05,
        "num_leaves": 15,
        "max_depth": 4,
        "seed": SEED + 1,
        "n_jobs": -1,
        "verbose": -1,
        "colsample_bytree": 0.8,
        "subsample": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
    }

    meta_oof_pred = np.zeros(len(meta_X))
    final_test_pred = np.zeros(len(meta_X_test))
    cv_iter = get_cv_splits(meta_X, y, n_splits=N_SPLITS, seed=SEED + 2, groups=groups)

    for tr_idx, va_idx in cv_iter:
        X_tr, y_tr = meta_X.iloc[tr_idx], y.iloc[tr_idx]
        X_va = meta_X.iloc[va_idx]

        meta_model = lgb.LGBMClassifier(**meta_lgb_params)
        meta_model.fit(X_tr, y_tr)

        meta_oof_pred[va_idx] = meta_model.predict_proba(X_va)[:, 1]
        final_test_pred += meta_model.predict_proba(meta_X_test)[:, 1] / N_SPLITS

    final_thr, _ = best_threshold(y, meta_oof_pred, metric="f1")
    final_f1 = f1_score(y, (meta_oof_pred >= final_thr).astype(int))

    print()
    print("========== Stacking CV Summary (LGB Meta-Model) ==========")
    print(f"Final Stacked OOF F1 (thr={final_thr:.3f}): {final_f1:.4f}")

    stacking_sub_path = LABEL_SUB_PATH.replace(".csv", "_stacking_lgbm_meta.csv")
    save_submission_label_noheader(test_ids, final_test_pred, stacking_sub_path, thr=final_thr)
    print()
    print(f"[INFO] Stacking submission saved to: {stacking_sub_path} (thr={final_thr:.3f})")
    print()
    print("🏆🏆🏆 Stacking pipeline finished successfully! 🏆🏆🏆")

    analyze_errors(X, y, meta_oof_pred, final_thr)


In [ ]:
def analyze_errors(X, y, oof_preds, thr):
    """
    モデルの予測エラーを分析し、間違えた予測の傾向を出力する
    """
    print("\n========== 🕵️  Starting Error Analysis 🕵️ ==========")

    # 分析用のデータフレームを作成
    df_analysis = X.copy()
    df_analysis['true_label'] = y
    df_analysis['predicted_proba'] = oof_preds
    df_analysis['predicted_label'] = (oof_preds >= thr).astype(int)

    # 間違えた予測を抽出
    errors = df_analysis[df_analysis['true_label'] != df_analysis['predicted_label']]

    # エラーの種類を特定
    # False Positive: 「デフォルトする(1)」と予測したが、実際はしなかった(0)
    fp = errors[(errors['true_label'] == 0) & (errors['predicted_label'] == 1)]
    # False Negative: 「デフォルトしない(0)」と予測したが、実際はした(1)
    fn = errors[(errors['true_label'] == 1) & (errors['predicted_label'] == 0)]

    print(f"Total errors: {len(errors)} / {len(df_analysis)}")
    print(f"  - False Positives (FP): {len(fp)}")
    print(f"  - False Negatives (FN): {len(fn)}")

    if len(errors) == 0:
        print("No errors found. Perfect model!")
        return

    print("\n--- Error Analysis for False Negatives (最も重要な間違い) ---")
    print("FN: デフォルトすると見抜けなかったローンの特徴:")

    # 間違いが多いカテゴリカル特徴量の傾向
    key_cat_cols = ['NaicsSector', 'BusinessAge', 'Subprogram']
    for col in key_cat_cols:
        if col in fn.columns:
            print(f"\n▼▼▼ {col} ▼▼▼")
            # 全データでの割合と、FNデータでの割合を比較
            df_comp = pd.DataFrame({
                'Overall %': df_analysis[col].value_counts(normalize=True) * 100,
                'FN %': fn[col].value_counts(normalize=True) * 100
            }).fillna(0).sort_values('FN %', ascending=False)

            print(df_comp.head(5))


    key_num_cols = ['InitialInterestRate', 'TermInMonths', 'GrossApproval', 'SBA_Portion']
    print("\n▼▼▼ Numerical Features (Mean) ▼▼▼")
    print(df_analysis[key_num_cols].describe().loc[['mean']].rename(index={'mean': 'Overall'}))
    print(fn[key_num_cols].describe().loc[['mean']].rename(index={'mean': 'False Negative'}))

In [ ]:
run_stacking_pipeline()

In [ ]:
def run_pseudo_labeling_pipeline(
    confidence_threshold_high=0.95,
    confidence_threshold_low=0.05,
):
    print("========== 🚀 Starting Final Pipeline: Maximizing Accuracy 🚀 ==========")

    print()
    print("--- [1/4] Generating high-accuracy predictions for pseudo-labeling ---")
    X, y, X_test, test_ids, _, cat_cols, target_col, _, groups = load_prepared_data(verbose=False)
    full_n_trials = 50

    lgb_result = train_lightgbm_model(
        X,
        y,
        X_test,
        cat_cols,
        n_trials=full_n_trials,
        groups=groups,
    )
    cb_result = train_catboost_model(
        X,
        y,
        X_test,
        cat_cols,
        n_trials=full_n_trials,
        groups=groups,
    )
    xgb_result = train_xgboost_model(
        X,
        y,
        X_test,
        n_trials=full_n_trials,
        groups=groups,
    )

    base_test_preds = np.mean(
        [lgb_result["test_pred"], cb_result["test_pred"], xgb_result["test_pred"]],
        axis=0,
    )
    print()
    print("[SUCCESS] High-accuracy predictions generated.")

    print()
    print("--- [2/4] Creating pseudo-labeled dataset ---")
    pseudo_labels = [
        1 if pred >= confidence_threshold_high else 0 if pred <= confidence_threshold_low else -1
        for pred in base_test_preds
    ]
    X_test_pseudo = X_test.copy()
    X_test_pseudo[target_col] = pseudo_labels
    X_test_pseudo_confident = X_test_pseudo[X_test_pseudo[target_col] != -1]

    print(f"Found {len(X_test_pseudo_confident)} confident pseudo-labels.")
    if len(X_test_pseudo_confident) < 100:
        print("WARNING: Not enough confident samples found. Aborting.")
        return

    X_with_pseudo = pd.concat([X, X_test_pseudo_confident.drop(columns=[target_col])], ignore_index=True)
    y_with_pseudo = pd.concat([y, X_test_pseudo_confident[target_col]], ignore_index=True)
    print(f"New training data size: {len(X_with_pseudo)} samples.")

    print()
    print("--- [3/4] Re-training models and creating individual submissions ---")
    orig_X_test = X_test.copy()
    orig_test_ids = test_ids.copy()
    orig_train_len = len(y)

    print()
    print("--- [Re-training 1/3] LightGBM ---")
    lgb_final = train_lightgbm_model(
        X_with_pseudo,
        y_with_pseudo,
        orig_X_test,
        cat_cols,
        n_trials=full_n_trials,
    )
    lgb_thr, lgb_f1 = best_threshold(
        y_with_pseudo[:orig_train_len],
        lgb_final["oof_pred"][:orig_train_len],
        metric="f1",
    )
    print(f"  -> Single LGBM OOF F1: {lgb_f1:.4f} (thr={lgb_thr:.3f})")
    save_submission_label_noheader(
        orig_test_ids,
        lgb_final["test_pred"],
        LABEL_SUB_PATH.replace(".csv", "_single_lgbm.csv"),
        thr=lgb_thr,
    )

    print()
    print("--- [Re-training 2/3] CatBoost ---")
    cb_final = train_catboost_model(
        X_with_pseudo,
        y_with_pseudo,
        orig_X_test,
        cat_cols,
        n_trials=full_n_trials,
    )
    cb_thr, cb_f1 = best_threshold(
        y_with_pseudo[:orig_train_len],
        cb_final["oof_pred"][:orig_train_len],
        metric="f1",
    )
    print(f"  -> Single CatBoost OOF F1: {cb_f1:.4f} (thr={cb_thr:.3f})")
    save_submission_label_noheader(
        orig_test_ids,
        cb_final["test_pred"],
        LABEL_SUB_PATH.replace(".csv", "_single_catboost.csv"),
        thr=cb_thr,
    )

    print()
    print("--- [Re-training 3/3] XGBoost ---")
    xgb_final = train_xgboost_model(
        X_with_pseudo,
        y_with_pseudo,
        orig_X_test,
        n_trials=full_n_trials,
    )
    xgb_thr, xgb_f1 = best_threshold(
        y_with_pseudo[:orig_train_len],
        xgb_final["oof_pred"][:orig_train_len],
        metric="f1",
    )
    print(f"  -> Single XGBoost OOF F1: {xgb_f1:.4f} (thr={xgb_thr:.3f})")
    save_submission_label_noheader(
        orig_test_ids,
        xgb_final["test_pred"],
        LABEL_SUB_PATH.replace(".csv", "_single_xgboost.csv"),
        thr=xgb_thr,
    )

    print()
    print("--- [4/4] Training final meta-model and creating stacking submission ---")
    meta_X_final = build_meta_features(
        lgb_pred=lgb_final["oof_pred"],
        cb_pred=cb_final["oof_pred"],
        xgb_pred=xgb_final["oof_pred"],
    )
    meta_X_test_final = build_meta_features(
        lgb_pred=lgb_final["test_pred"],
        cb_pred=cb_final["test_pred"],
        xgb_pred=xgb_final["test_pred"],
    )

    meta_model_final = LogisticRegression(class_weight="balanced", random_state=SEED, C=0.1)
    meta_model_final.fit(meta_X_final, y_with_pseudo)

    final_test_pred = meta_model_final.predict_proba(meta_X_test_final)[:, 1]
    final_oof_pred = meta_model_final.predict_proba(meta_X_final)[:, 1]

    final_thr, final_f1 = best_threshold(
        y_with_pseudo[:orig_train_len],
        final_oof_pred[:orig_train_len],
        metric="f1",
    )
    print()
    print("========== Final Stacking Model CV Summary ==========")
    print(f"Final Stacked OOF F1 on original data (thr={final_thr:.3f}): {final_f1:.4f}")

    final_sub_path = LABEL_SUB_PATH.replace(".csv", "_final_stacking.csv")
    save_submission_label_noheader(orig_test_ids, final_test_pred, final_sub_path, thr=final_thr)
    print()
    print(f"[INFO] Final Stacking submission saved to: {final_sub_path}")
    print()
    print("🏆🏆🏆 All 4 submission files created! This is our final attack! 🏆🏆🏆")


In [ ]:
run_pseudo_labeling_pipeline()

========== 🚀 Starting Final Pipeline: Maximizing Accuracy 🚀 ==========

--- [1/4] Generating high-accuracy predictions for pseudo-labeling ---
[INFO] Creating final aggregation and interaction features...


[I 2025-08-29 00:37:48,854] A new study created in memory with name: no-name-11ee4237-16bc-4468-b637-be0598e59783


[SUCCESS] Added 96 aggregation features!
[INFO] class balance: pos=964, neg=6588, base_pos_weight≈6.834
[INFO] threshold metric = f1


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-08-29 00:38:14,644] Trial 0 finished with value: 0.6318573495154991 and parameters: {'learning_rate': 0.042472407130841744, 'num_leaves': 223, 'min_child_samples': 149, 'feature_fraction': 0.8394633936788146, 'bagging_fraction': 0.6624074561769746, 'bagging_freq': 1, 'lambda_l1': 3.3323645788192616e-08, 'lambda_l2': 0.6245760287469893, 'min_split_gain': 0.6011150117432088, 'max_depth': 8, 'pos_weight_mult': 0.5720457300353086}. Best is trial 0 with value: 0.6318573495154991.
[I 2025-08-29 00:38:27,438] Trial 1 finished with value: 0.6337116103188076 and parameters: {'learning_rate': 0.07819459112971966, 'num_leaves': 160, 'min_child_samples': 50, 'feature_fraction': 0.6727299868828402, 'bagging_fraction': 0.6733618039413735, 'bagging_freq': 3, 'lambda_l1': 0.00052821153945323, 'lambda_l2': 7.71800699380605e-05, 'min_split_gain': 0.2912291401980419, 'max_depth': 7, 'pos_weight_mult': 0.9882285122821464}. Best is trial 1 with value: 0.6337116103188076.
[I 2025-08-29 00:38:55,887]

...


========== [CatBoost] Best Trial ==========
F1 (CV mean): 0.8992
Best params:
  iterations: 1719
  learning_rate: 0.0283975330845199
  depth: 6
  l2_leaf_reg: 5.94789434145484
  random_strength: 0.01352866335469214
  bagging_temperature: 0.7717297583062844
  border_count: 233
[INFO][CatBoost] Starting final CV training and prediction...

========== [CatBoost] CV Summary ==========
OOF AUC: 0.9520
OOF F1 (thr=0.690): 0.6434
  -> Single CatBoost OOF F1: 0.6234 (thr=0.690)


[I 2025-08-29 02:51:22,005] A new study created in memory with name: no-name-f05a0fb3-e579-4c9f-9e29-9be3fe9581d1


[INFO] headerless label submission saved to: /content/drive/MyDrive/MUFG/submission_1_single_catboost.csv (thr=0.690)

--- [Re-training 3/3] XGBoost ---
[INFO][XGBoost] Starting hyperparameter optimization...


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2025-08-29 02:51:42,763] Trial 0 finished with value: 0.9518435476437382 and parameters: {'lambda': 0.012344123349944376, 'alpha': 0.012774073536792349, 'max_depth': 6, 'eta': 0.05265951583759404, 'gamma': 5.002752869965208e-08, 'colsample_bytree': 0.9209815685503764, 'subsample': 0.7063351732356111, 'min_child_weight': 7}. Best is trial 0 with value: 0.9518435476437382.
[I 2025-08-29 02:52:06,245] Trial 1 finished with value: 0.9519821740677546 and parameters: {'lambda': 0.011061232626392132, 'alpha': 0.023074977517065037, 'max_depth': 6, 'eta': 0.04614603588568411, 'gamma': 7.059919087061035e-06, 'colsample_bytree': 0.5645416732267381, 'subsample': 0.8561340742313805, 'min_child_weight': 4}. Best is trial 1 with value: 0.9519821740677546.
[I 2025-08-29 02:52:18,391] Trial 2 finished with value: 0.9530238605162864 and parameters: {'lambda': 0.012110484113478908, 'alpha': 0.02028067369215875, 'max_depth': 4, 'eta': 0.07487271671280067, 'gamma': 0.07148849023002905, 'colsample_bytree

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score


def perform_adversarial_validation(train_path, test_path):
    """
    Adversarial Validationを実行して、訓練データとテストデータの分布の違いを評価する
    """
    print("========== 🕵️  Starting Adversarial Validation 🕵️ ==========")

    train = pd.read_csv(train_path)
    test = pd.read_csv(test_path)

    train["is_test"] = 0
    test["is_test"] = 1

    target_col = "LoanStatus"
    id_cols = [col for col in train.columns if "id" in col.lower()]
    cols_to_drop = [target_col, "is_test", *id_cols]

    combined = pd.concat([train, test], axis=0, ignore_index=True)
    features = [col for col in combined.columns if col not in cols_to_drop]

    for col in features:
        if combined[col].dtype == "object":
            combined[col] = combined[col].astype("category").cat.codes

    X = combined[features]
    y = combined["is_test"]

    lgb_params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "n_estimators": 500,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "seed": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(combined))
    feature_importances = pd.DataFrame(index=features)

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
        X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        model = lgb.LGBMClassifier(**lgb_params)
        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
        )

        oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
        feature_importances[f"fold_{fold}"] = model.feature_importances_

    auc_score = roc_auc_score(y, oof_preds)
    print()
    print(f"[Result] Adversarial Validation AUC: {auc_score:.4f}")
    print()

    if auc_score > 0.7:
        print("AUC > 0.7: 訓練データとテストデータの分布に significant な違いがあります。")
        print("CV戦略の見直しや、以下の特徴量の扱いを検討する必要があります。")
    elif auc_score > 0.6:
        print("AUC > 0.6: 訓練データとテストデータの分布に若干の違いが見られます。")
    else:
        print("AUC < 0.6: 訓練データとテストデータの分布は非常に似ています。")

    feature_importances["mean"] = feature_importances.mean(axis=1)
    top_features = feature_importances.sort_values("mean", ascending=False).head(15)

    print()
    print("--- Top 15 features to distinguish Train vs Test ---")
    print(top_features["mean"])


perform_adversarial_validation(TRAIN_PATH, TEST_PATH)


========== 🕵️  Starting Adversarial Validation 🕵️ ==========

[Result] Adversarial Validation AUC: 0.4970

AUC < 0.6: 訓練データとテストデータの分布は非常に似ています。

--- Top 15 features to distinguish Train vs Test ---
InitialInterestRate           52.6
SBAGuaranteedApproval         48.0
GrossApproval                 43.0
TermInMonths                  34.2
JobsSupported                 27.6
CongressionalDistrict         25.6
NaicsSector                   22.8
ApprovalFiscalYear             7.6
Subprogram                     7.4
BusinessAge                    7.0
BusinessType                   3.8
CollateralInd                  3.4
RevolverStatus                 2.8
FixedOrVariableInterestInd     2.2
Name: mean, dtype: float64
